# ⛓️ หน่วยที่ 3 — LangChain Chain (LCEL)

## Chain คืออะไร?
**Chain = ต่อท่อ** → ผลจากขั้นที่ 1 ไหลไปขั้นที่ 2 อัตโนมัติ

```
Prompt → LLM → OutputParser
   |       |        |
  สร้าง   ส่ง AI   แปลงผล
  คำถาม   ตอบ      ให้สวย
```

LangChain ใช้ `|` (pipe) ต่อกัน เรียกว่า **LCEL** (LangChain Expression Language)

## โครงสร้าง:
```
main()                          ← หลัก
├── setup_llm()                 ← รอง
├── demo_simple_chain()         ← รอง: Chain แบบง่าย
├── demo_prompt_template()      ← รอง: PromptTemplate
├── demo_output_parser()        ← รอง: แปลงผลลัพธ์
├── demo_trading_chain()        ← รอง: Chain สำหรับเทรด
└── demo_chain_vs_agent()       ← รอง: Chain vs Agent
```

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
print(f'✅ Model: {OPENAI_MODEL}')

## 🔧 ฟังก์ชันรอง 1: `setup_llm()`

In [ ]:
def setup_llm(temperature=0.1):
    return ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=temperature)

# เรียกใช้
llm = setup_llm()
print('✅ LLM พร้อม')

## ⛓️ ฟังก์ชันรอง 2: `demo_simple_chain()`

**Chain แบบง่ายที่สุด** → `prompt | llm | parser`

เครื่องหมาย `|` (pipe) คือ "ส่งต่อ" เหมือนท่อน้ำ:
- `prompt` สร้างคำถาม → ส่งให้ `llm` → ส่งให้ `parser` แปลงผล

In [ ]:
def demo_simple_chain(llm):
    """Chain แบบง่าย: prompt | llm | parser"""
    print('\n⛓️ === Simple Chain ===')

    # สร้าง Chain 3 ขั้น
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'ตอบสั้นๆ 1-2 บรรทัด'),
        ('human', '{question}'),
    ])
    parser = StrOutputParser()  # แปลง AIMessage → string

    chain = prompt | llm | parser  # ← นี่คือ Chain!

    # เรียกใช้ Chain
    result = chain.invoke({'question': 'LangChain คืออะไร?'})
    print(f'🤖: {result}')
    print(f'📦 type = {type(result).__name__}')  # str ไม่ใช่ AIMessage

    # เปรียบเทียบกับไม่ใช้ Chain
    print('\n💡 เปรียบเทียบ:')
    print(f'   llm.invoke()  → type = {type(llm.invoke("สวัสดี")).__name__}')  # AIMessage
    print(f'   chain.invoke() → type = {type(result).__name__}')          # str
    print('💡 Chain ทำให้ได้ string ตรงๆ ไม่ต้อง .content!')

# เรียกใช้
demo_simple_chain(llm)

## 📝 ฟังก์ชันรอง 3: `demo_prompt_template()`

`ChatPromptTemplate` = **template** ที่เปลี่ยนค่าได้ เหมือน f-string

ข้อดี: สร้าง prompt 1 template → ใช้กับหลายคำถามได้

In [ ]:
def demo_prompt_template(llm):
    """PromptTemplate — สร้าง prompt template ที่ reusable"""
    print('\n📝 === PromptTemplate ===')

    # template 1 ตัว → ใช้กับหลายสินทรัพย์
    template = ChatPromptTemplate.from_messages([
        ('system', 'คุณเป็นนักวิเคราะห์ {asset_type} ตอบสั้นๆ'),
        ('human', '{asset} ราคา {price} แนวโน้มเป็นอย่างไร?'),
    ])
    chain = template | llm | StrOutputParser()

    # ใช้ template เดิม เปลี่ยนแค่ตัวแปร
    r1 = chain.invoke({'asset_type': 'ทองคำ', 'asset': 'XAUUSD', 'price': '2650'})
    print(f'🥇 ทอง: {r1}')

    r2 = chain.invoke({'asset_type': 'หุ้น', 'asset': 'AAPL', 'price': '180'})
    print(f'📱 หุ้น: {r2}')

    print('💡 template เดียว ใช้ได้กับทุกสินทรัพย์!')

# เรียกใช้
demo_prompt_template(llm)

## 🔄 ฟังก์ชันรอง 4: `demo_output_parser()`

OutputParser = แปลงผลลัพธ์จาก AI ให้อยู่ในรูปแบบที่ต้องการ

ตัวอย่าง: สั่ง AI ตอบเป็น JSON → parse เป็น dict ให้เลย

In [ ]:
import json

def demo_output_parser(llm):
    """OutputParser — แปลงผลลัพธ์จาก AI"""
    print('\n🔄 === OutputParser ===')

    template = ChatPromptTemplate.from_messages([
        ('system', 'ตอบเป็น JSON เท่านั้น ห้ามใส่ markdown คำอธิบาย หรือ ```'),
        ('human', 'วิเคราะห์ {asset} ราคา {price} ตอบ JSON: '
                  '{{"trend": "up/down/sideways", "action": "BUY/SELL/HOLD", "reason": "..."}}'),
    ])
    chain = template | llm | StrOutputParser()

    raw = chain.invoke({'asset': 'XAUUSD', 'price': '2650'})
    print(f'📨 Raw: {raw}')

    try:
        data = json.loads(raw)
        print(f'✅ Parsed dict:')
        print(f'   trend  = {data["trend"]}')
        print(f'   action = {data["action"]}')
        print(f'   reason = {data["reason"]}')
    except json.JSONDecodeError:
        print(f'⚠️ AI ไม่ตอบ JSON ตรง → ต้องปรับ prompt')

# เรียกใช้
demo_output_parser(llm)

## 📊 ฟังก์ชันรอง 5: `demo_trading_chain()`

ต่อ Chain **2 ขั้น** สำหรับเทรด:
1. Chain 1: วิเคราะห์เทรนด์
2. Chain 2: ส่งเทรนด์ → สร้างแผนเทรด

In [ ]:
def demo_trading_chain(llm):
    """Chain 2 ขั้นสำหรับเทรด: วิเคราะห์ → แผนเทรด"""
    print('\n📊 === Trading Chain (2 ขั้น) ===')

    # Chain 1: วิเคราะห์เทรนด์
    trend_prompt = ChatPromptTemplate.from_messages([
        ('system', 'วิเคราะห์เทรนด์สั้นๆ 1 บรรทัด'),
        ('human', '{asset} ราคาปัจจุบัน {price} เทรนด์เป็นอย่างไร?'),
    ])
    trend_chain = trend_prompt | llm | StrOutputParser()
    trend = trend_chain.invoke({'asset': 'XAUUSD', 'price': '2650'})
    print(f'📈 ขั้น 1 (เทรนด์): {trend}')

    # Chain 2: สร้างแผนเทรด (รับผลจาก Chain 1)
    plan_prompt = ChatPromptTemplate.from_messages([
        ('system', 'สร้างแผนเทรดจากผลวิเคราะห์ | ตอบ: Entry, SL, TP, R:R'),
        ('human', 'ผลวิเคราะห์: {trend_analysis}\nสร้างแผนเทรดหน่วย pip'),
    ])
    plan_chain = plan_prompt | llm | StrOutputParser()
    plan = plan_chain.invoke({'trend_analysis': trend})
    print(f'\n📋 ขั้น 2 (แผนเทรด):\n{plan}')

    print('\n💡 Chain ทำทีละขั้น: วิเคราะห์ก่อน → แผนเทรดทีหลัง')

# เรียกใช้
demo_trading_chain(llm)

## ⚡ ฟังก์ชันรอง 6: `demo_chain_vs_agent()`

| | Chain | Agent |
|---|-------|-------|
| **ลำดับ** | ตายตัว (1→2→3) | AI ตัดสินใจเอง |
| **Tool** | ไม่มี | มี |
| **ใช้เมื่อ** | รู้ขั้นตอนแน่นอน | ไม่รู้ล่วงหน้าว่าต้องทำอะไร |

In [ ]:
def demo_chain_vs_agent(llm):
    """เปรียบเทียบ Chain vs Agent"""
    print('\n⚡ === Chain vs Agent ===')
    print()
    print('⛓️ Chain:')
    print('   prompt | llm | parser')
    print('   ลำดับตายตัว: 1→2→3 เสมอ')
    print('   เหมาะกับ: งานที่รู้ขั้นตอนแน่นอน')
    print()
    print('🤖 Agent:')
    print('   AgentExecutor(agent, tools)')
    print('   AI คิดเอง → เลือก Tool → ดูผล → คิดต่อ')
    print('   เหมาะกับ: งานที่ไม่รู้ล่วงหน้าว่าจะต้องทำอะไร')
    print()
    print('💡 หน่วยที่ 4 จะสอน Agent!')

# เรียกใช้
demo_chain_vs_agent(llm)

---
## ★ ฟังก์ชันหลัก `main()` — รันทุกอย่างรวมกัน

In [ ]:
def main():
    llm = setup_llm()                    # รอง 1
    demo_simple_chain(llm)               # รอง 2
    demo_prompt_template(llm)            # รอง 3
    demo_output_parser(llm)              # รอง 4
    demo_trading_chain(llm)              # รอง 5
    demo_chain_vs_agent(llm)             # รอง 6
    print('\n✅ จบหน่วยที่ 3!')

# เรียกใช้
main()